### Install Libraries

In [11]:
!pip install ddgs trafilatura

### Setup

In [12]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import json
from pprint import pprint
from ddgs import DDGS
import trafilatura

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

client = OpenAI()

### Step 1: Define the Tools

In [26]:
def search_web(query: str):
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f"  \u2705 Got Results")
    return json.dumps(results, indent=2)

In [27]:
#'beautiful soup' is another package you can use if you need more control

def fetch_url(url: str):
    downloaded = trafilatura.fetch_url(url)

    if downloaded:
        text = trafilatura.extract(downloaded)
        print(f"  \n \u2705 Got text: {len(text)} chars")
        return text
    else:
        print(f"  \u274C Failed to fetch or extract text from {url}")
        return f"Could not extract text from {url}. Try a different source."

In [30]:
results = search_web("AI in healthcare in 2030")
print(results)

  ✅ Got Results
[
  {
    "title": "Pulse Report: Healthcare in 2030: How AI Agents Promote Integrated ...",
    "href": "https://www.himss.org/resources/pulse-report-healthcare-in-2030-how-ai-agents-promote-integrated-collaborative-and-proactive-care/",
    "body": "With the rapid democratization of AI, the future of healthcare was already under redevelopment. Now, in an era of AI agents, that future is getting reshaped. What might this new healthcare future, powered by AI agents, look like? Imagine analyzing the most complex, distributed data across vast ecosystems to forecast possible health events or predict surges in demand, activating both mitigation ..."
  },
  {
    "title": "AI in Healthcare: 3 Revolutionary Trends by 2030 | Kindo",
    "href": "https://www.kindo.ai/blog/how-ai-will-change-healthcare-by-2030",
    "body": "AI transforms healthcare via predictive analytics, precision medicine, and automated diagnostics. Explore 3 key shifts defining the 2030 medical landscape."

In [31]:
fetch_results = fetch_url("https://www.himss.org/resources/pulse-report-healthcare-in-2030-how-ai-agents-promote-integrated-collaborative-and-proactive-care/")
print(fetch_results)

  ❌ Failed to fetch or extract text from https://www.himss.org/resources/pulse-report-healthcare-in-2030-how-ai-agents-promote-integrated-collaborative-and-proactive-care/
Could not extract text from https://www.himss.org/resources/pulse-report-healthcare-in-2030-how-ai-agents-promote-integrated-collaborative-and-proactive-care/. Try a different source.


In [32]:
tools = []

In [ ]:
search_web_function = {
    "name": "search_web",
    "description": "Search the web using DuckDuckGo browser. Returns 3 results",
    "parameters" : {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to find relevant websites."
            }
        },
        "required": ["query"]
    }
}

fetch_url_function = {
    "name": "fetch_url",
    "description": "Fetch and extract the main text content from a web page",
    "parameters" : {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "The URL of the web page to fetch and extract text from"
            }
        },
        "required": ["url"]
    }
}

tools.append({type: "function", "function": fetch_url_function})

In [ ]:

def handle_tool_calls(tool_calls):
    tool_call_results = []

    for tool_call in tool_calls:
        if tool_call.function.name == "search_web":
            args = json.loads(tool_call.function.arguments) 
            print(f"  \U0001f527 Calling function {function_name} with arguments: {args}") #for debugging
            result = search_web(args["query"])
            content = f"Search Result: {result}"
        elif tool_call.function.name == "fetch_url":
            result = fetch_url(args["url"])
            content = f'"Fetched URL Content: {result}'
        else:
            content = f"Unknown tool call: {tool_call.function.name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        }
            
        tool_call_results.append(tool_call_result)
        
    return tool_call_results

[{type: 'function',
  'function': {'name': 'search_web',
   'description': 'Search the web using DuckDuckGo browser. Returns 3 results',
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string',
      'description': 'The search query to find relevant websites.'}},
    'required': ['query']}}},
 {type: 'function',
  'function': {'name': 'fetch_url',
   'description': 'Fetches/extracts text from a supplied url',
   'parameters': {'type': 'object',
    'properties': {'url': {'type': 'string',
      'description': 'The url to be used to extract text.'}},
    'required': ['url']}}}]